# AUTO-LABELING TOOL — Keyword-Based Sentiment Scoring
## Sentiment Analysis of Indonesian Political News

**Metodologi:** Lexicon-Based Sentiment Scoring  
**Input:** Raw articles (CNBC, Detik, Kompas)  
**Output:** Labeled dataset (-1 Negatif / 0 Netral / 1 Positif)

### Cara Kerja:
1. Setiap artikel dihitung berapa kata positif dan negatif yang muncul
2. Kalau kata positif > negatif → label **Positif (1)**
3. Kalau kata negatif > positif → label **Negatif (-1)**  
4. Kalau seimbang/tidak ada → label **Netral (0)**

**Referensi:** Taboada et al. (2011). Lexicon-Based Methods for Sentiment Analysis. *Computational Linguistics*, 37(2), 267-307.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from collections import Counter

print('Libraries loaded!')

In [ ]:
# ================================================================
# DOMAIN-SPECIFIC POLITICAL SENTIMENT LEXICON
# Dibangun berdasarkan karakteristik bahasa jurnalistik politik Indonesia
# ================================================================

POS_KW = [
    # Stabilitas & Keamanan
    'stabil', 'kondusif', 'damai', 'rukun', 'aman', 'tertib',
    'rekonsiliasi', 'harmonis', 'sinergi', 'solid', 'bersatu',
    'kompak', 'gotong royong', 'musyawarah',

    # Kinerja & Ekonomi
    'surplus', 'tumbuh', 'menguat', 'pulih', 'efektif', 'efisien',
    'terobosan', 'inovasi', 'prestasi', 'capaian', 'rampung',
    'diresmikan', 'investasi masuk', 'elektabilitas naik',
    'meningkat', 'naik', 'bertumbuh', 'berkembang', 'maju',
    'berhasil', 'sukses', 'optimal', 'produktif', 'unggul',

    # Dukungan Politik & Hukum
    'apresiasi', 'dukungan', 'sepakat', 'setuju', 'koalisi',
    'resmi', 'sah', 'transparan', 'akuntabel',
    'amanah', 'bersih', 'bebas korupsi', 'konstitusional',
    'demokratis', 'legitimate',

    # Pembangunan & Kebijakan Positif
    'pembangunan', 'infrastruktur', 'program', 'kebijakan baik',
    'reformasi', 'perbaikan', 'kemajuan', 'kesejahteraan',
    'bantuan', 'subsidi', 'stimulus', 'insentif',
    'penghargaan', 'award', 'juara', 'terbaik'
]

NEG_KW = [
    # Konflik & Ketidakstabilan
    'konflik', 'gejolak', 'ricuh', 'anarkis', 'bentrok', 'tegang',
    'pecah kongsi', 'kudeta', 'makar', 'teror', 'ancaman',
    'provokasi', 'adu domba', 'radikal', 'separatis', 'krisis', 'skandal',
    'kerusuhan', 'chaos', 'kacau', 'rusuh',

    # Masalah Hukum & Korupsi
    'korupsi', 'suap', 'gratifikasi', 'pungli', 'tersangka', 'terdakwa',
    'buron', 'sanksi', 'pelanggaran', 'ilegal', 'cacat hukum',
    'manipulasi', 'penggelapan', 'penyelewengan', 'ott',
    'ditangkap', 'ditahan', 'divonis', 'dihukum', 'dipenjara',
    'dakwaan', 'tuntutan', 'sidang', 'vonis', 'hukuman',

    # Kinerja Buruk & Kritik
    'gagal', 'mangkrak', 'anjlok', 'rugi', 'resesi', 'utang membengkak',
    'pengangguran', 'ketimpangan', 'kritik', 'kecam', 'protes',
    'demo', 'menolak', 'mosi tidak percaya', 'mundur', 'copot',
    'blunder', 'kelalaian', 'tekanan', 'tertekan',
    'melemah', 'turun', 'menurun', 'jatuh', 'merosot',
    'deflasi', 'inflasi tinggi', 'defisit',

    # Politik Negatif
    'pecat', 'dicopot', 'dipecat', 'diberhentikan', 'dilengserkan',
    'ditolak', 'diboikot', 'blacklist', 'diblokir',
    'sengketa', 'perselisihan', 'perseteruan', 'pertikaian',
    'bocor', 'kebocoran', 'penyadapan',
    'intimidasi', 'tekanan', 'pemaksaan',

    # Bencana & Krisis
    'bencana', 'gempa', 'banjir', 'kebakaran', 'musibah',
    'korban', 'meninggal', 'tewas', 'tragedi'
]

print(f'Positive keywords: {len(POS_KW)}')
print(f'Negative keywords: {len(NEG_KW)}')

In [ ]:
# ================================================================
# CORE LABELING FUNCTION
# ================================================================

def score_text(text):
    """Hitung skor sentimen berdasarkan keyword matching."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0, 0
    
    t = text.lower()
    pos_score = sum(1 for kw in POS_KW if kw in t)
    neg_score = sum(1 for kw in NEG_KW if kw in t)
    return pos_score, neg_score

def assign_label(text):
    """Tentukan label sentimen berdasarkan skor keyword."""
    pos, neg = score_text(text)
    
    if pos > neg and pos > 0:
        return 1    # Positif
    elif neg > pos and neg > 0:
        return -1   # Negatif
    else:
        return 0    # Netral

def label_dataset(df, text_col='text', verbose=True):
    """Label seluruh dataset menggunakan keyword scoring."""
    df = df.copy()
    df['label'] = df[text_col].apply(assign_label)
    
    if verbose:
        dist = df['label'].value_counts().sort_index()
        total = len(df)
        for lbl, cnt in dist.items():
            name = {-1:'Negatif', 0:'Netral', 1:'Positif'}[lbl]
            print(f'  {name:10}: {cnt:,} ({cnt/total*100:.1f}%)')
    
    return df

# Test dengan contoh
examples = [
    'Presiden berhasil dorong pertumbuhan ekonomi 5 persen',
    'Korupsi merajalela, pejabat ditetapkan tersangka oleh KPK',
    'Rapat kabinet membahas RAPBN 2027 hari ini di istana'
]
print('=== Test Labeling ===')
for ex in examples:
    lbl = assign_label(ex)
    name = {-1:'NEGATIF', 0:'NETRAL', 1:'POSITIF'}[lbl]
    print(f'  [{name}] {ex[:60]}')

In [ ]:
# ================================================================
# LOAD RAW DATA
# ================================================================

# Sesuaikan path dengan lokasi data kamu
DATA_PATH = '../../data/data_berita/raw_fixed/'
OUTPUT_PATH = '../../data/data_berita/cleaning/data_labeled/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

df_cnbc   = pd.read_csv(f'{DATA_PATH}/cnbc_articles.csv')
df_detik  = pd.read_csv(f'{DATA_PATH}/detik_articles.csv')
df_kompas = pd.read_csv(f'{DATA_PATH}/kompas_articles.csv')

print(f'CNBC  : {len(df_cnbc):,} artikel')
print(f'Detik : {len(df_detik):,} artikel')
print(f'Kompas: {len(df_kompas):,} artikel')
print(f'Total : {len(df_cnbc)+len(df_detik)+len(df_kompas):,} artikel')

In [ ]:
# ================================================================
# PREPROCESSING — Gabung title + content → text
# ================================================================

def preprocess(df, source_name):
    df = df.copy()
    
    # Rename kolom kalau perlu
    if 'published_at' in df.columns:
        df = df.rename(columns={'published_at': 'date'})

    # Gabung title + content jadi text
    df['text'] = (df['title'].fillna('') + '. ' + df['content'].fillna('')).str.strip()

    # Buat article_id
    prefix = source_name[:3].upper()
    df['article_id'] = [f'{prefix}_{str(i).zfill(5)}' for i in range(len(df))]

    # Pilih kolom
    df = df[['date', 'title', 'content', 'article_id', 'text']].copy()
    df = df.dropna(subset=['text'])
    df = df[df['text'].str.len() > 50]
    
    return df.reset_index(drop=True)

df_cnbc_clean   = preprocess(df_cnbc,   'cnbc')
df_detik_clean  = preprocess(df_detik,  'detik')
df_kompas_clean = preprocess(df_kompas, 'kompas')

print('Preprocessing selesai!')
print(f'CNBC  : {len(df_cnbc_clean):,}')
print(f'Detik : {len(df_detik_clean):,}')
print(f'Kompas: {len(df_kompas_clean):,}')

In [ ]:
# ================================================================
# AUTO LABELING — Semua Artikel
# ================================================================

print('=== LABELING CNBC ===')
df_cnbc_labeled = label_dataset(df_cnbc_clean)

print('\n=== LABELING DETIK ===')
df_detik_labeled = label_dataset(df_detik_clean)

print('\n=== LABELING KOMPAS ===')
df_kompas_labeled = label_dataset(df_kompas_clean)

# Distribusi gabungan
df_all = pd.concat([df_cnbc_labeled, df_detik_labeled, df_kompas_labeled])
print('\n=== DISTRIBUSI FINAL ===')
dist = df_all['label'].value_counts().sort_index()
for lbl, cnt in dist.items():
    name = {-1:'Negatif', 0:'Netral', 1:'Positif'}[lbl]
    print(f'  {name:10}: {cnt:,} ({cnt/len(df_all)*100:.1f}%)')
print(f'  Total     : {len(df_all):,}')

In [ ]:
# ================================================================
# SAVE HASIL LABELING
# ================================================================

df_cnbc_labeled.to_csv(f'{OUTPUT_PATH}/cnbc_labeled.csv',   index=False, encoding='utf-8-sig')
df_detik_labeled.to_csv(f'{OUTPUT_PATH}/detik_labeled.csv',  index=False, encoding='utf-8-sig')
df_kompas_labeled.to_csv(f'{OUTPUT_PATH}/kompas_labeled.csv', index=False, encoding='utf-8-sig')

print('=== SAVED ===')
print(f'cnbc_labeled.csv   → {len(df_cnbc_labeled):,} artikel')
print(f'detik_labeled.csv  → {len(df_detik_labeled):,} artikel')
print(f'kompas_labeled.csv → {len(df_kompas_labeled):,} artikel')
print(f'TOTAL              → {len(df_all):,} artikel')

## ✅ Selesai!

### Cara Kerja Tool ini:
- **Input**: Artikel mentah (title + content)
- **Proses**: Hitung berapa kata dari lexicon positif/negatif yang muncul
- **Output**: Label -1 (Negatif) / 0 (Netral) / 1 (Positif)

### Kenapa Keyword-Based?
1. **Interpretable** — bisa dijelaskan secara eksplisit
2. **Reproducible** — keyword list bisa diaudit
3. **Domain-specific** — lexicon dibangun khusus untuk berita politik Indonesia
4. **Efficient** — bisa label 25,000+ artikel dalam hitungan detik

### Referensi:
- Taboada et al. (2011). Lexicon-Based Methods for Sentiment Analysis. *Computational Linguistics*
- Turney (2002). Thumbs Up or Thumbs Down? Semantic Orientation Applied to Unsupervised Classification of Reviews. *ACL 2002*